# Band-edge settle shelf

This notebook follows the `1.20–1.30 R_s` spacing band after the spacing-boundary and loop-gain-retuning notes. The key question is whether the half-sine lane starts catching up as soon as it becomes track-ready again, or whether the threshold metric and the residual metric keep disagreeing for a while.

## Metrics

For each spacing/gain pair we keep two reads:

- **settle fraction**: the fraction of the last eight loop blocks that stay inside $\pm 0.05 R_s$
- **residual ratio**: $\rho = \frac{\text{half-sine mean tail residual}}{\text{proxy mean tail residual}}$

A loop can look recovered on the first metric while still lagging badly on the second.

In [ ]:
from __future__ import annotations

import csv
import sys
from pathlib import Path

repo = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
scripts = repo / 'scripts'
if str(scripts) not in sys.path:
    sys.path.insert(0, str(scripts))

from waveform_carrier_front_ends import band_edge_closed_loop_row

csv_path = repo / 'assets/2026-05-27-band-edge-settle-shelf.csv'
rows = []
with csv_path.open() as handle:
    for row in csv.DictReader(handle):
        row['channel_spacing'] = float(row['channel_spacing'])
        row['loop_gain'] = float(row['loop_gain'])
        row['proxy_tail_within_threshold_fraction'] = float(row['proxy_tail_within_threshold_fraction'])
        row['half_sine_tail_within_threshold_fraction'] = float(row['half_sine_tail_within_threshold_fraction'])
        row['residual_ratio_half_to_proxy'] = float(row['residual_ratio_half_to_proxy'])
        rows.append(row)

len(rows)


In [ ]:
baseline = [row for row in rows if abs(row['loop_gain'] - 0.020) < 1e-12]
baseline.sort(key=lambda row: row['channel_spacing'])
for row in baseline:
    print(
        f"spacing={row['channel_spacing']:.2f}  "
        f"settle={row['half_sine_tail_within_threshold_fraction']:.3f}  "
        f"ratio={row['residual_ratio_half_to_proxy']:.1f}x"
    )


In [ ]:
seed_pairs = [(19, 173), (23, 211), (31, 271), (47, 389)]
spacings = [1.20, 1.22, 1.24, 1.26, 1.28, 1.30]

for desired_seed, adjacent_seed in seed_pairs:
    ratios = []
    settles = []
    for spacing in spacings:
        proxy = band_edge_closed_loop_row(
            'proxy_bandpass',
            adjacent_enabled=True,
            adjacent_relative_power_db=0.0,
            desired_seed=desired_seed,
            adjacent_seed=adjacent_seed,
            channel_spacing=spacing,
            loop_gain=0.020,
        )
        half = band_edge_closed_loop_row(
            'gnuradio_half_sine',
            adjacent_enabled=True,
            adjacent_relative_power_db=0.0,
            desired_seed=desired_seed,
            adjacent_seed=adjacent_seed,
            channel_spacing=spacing,
            loop_gain=0.020,
        )
        ratios.append(half.tail_mean_abs_residual_cfo / proxy.tail_mean_abs_residual_cfo)
        settles.append(half.tail_within_threshold_fraction)
    print((desired_seed, adjacent_seed), [round(value, 1) for value in ratios], settles)


## What to notice

1. Lower gain really does reopen the settle band earlier.
2. That does **not** mean the half-sine lane is catching up in residual CFO.
3. The proxy lane stays fully settled across the tested grid, so this is not a case where both loops are equally fragile.

### Two short exercises

- Check the first spacing where the half-sine lane reaches a full settle fraction at gain `0.020`.
- Replace the gain with `0.010` in the seed sweep and see whether the residual ratio still rises with spacing for all four seed pairs.

### Caveat

This notebook still lives in the bounded detector-and-loop lane. It does not add timing recovery, BER, or a full modem chain.